# 2. Built-in AI governance and visibility

Use the same three settings as notebook 1: `AI_GATEWAY_CHAT_URL`, `AI_GATEWAY_API_KEY`, and `AI_GATEWAY_MODEL_ALIAS` in the root `.env` or shell environment. Shell values take precedence. This notebook sends at most **three** short requests. To see the gateway reject excess use, first configure a small **model consumption limit** for an isolated demo caller in the [AI Gateway portal documentation](https://ai.gateway.azure.com/docs); make the threshold low enough to trigger within three requests. Otherwise all three may succeed. A backend Foundry quota failure is not proof of a gateway policy.

**Demo:** Run the cells, show each HTTP status and any reported token counts, then compare them with gateway policy and telemetry. A `429` requires checking its source; do not change limits on a shared production caller.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv

lab_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(lab_root / '.env', override=False)

chat_url = os.getenv('AI_GATEWAY_CHAT_URL', '').strip()
api_key = os.getenv('AI_GATEWAY_API_KEY', '').strip()
alias = os.getenv('AI_GATEWAY_MODEL_ALIAS', '').strip()
if not all((chat_url, api_key, alias)):
    raise RuntimeError('Set AI_GATEWAY_CHAT_URL, AI_GATEWAY_API_KEY, and AI_GATEWAY_MODEL_ALIAS in the root .env or shell environment.')
parsed = urlparse(chat_url)
if parsed.scheme != 'https' or not parsed.netloc or not parsed.path.endswith('/chat/completions'):
    raise ValueError('Use the full HTTPS Chat completions endpoint copied from the model Overview in AI Gateway.')
print('Testing alias:', alias, 'via:', parsed.hostname)


## Measure request usage and observe an optional limit

Successful responses may include provider-reported token usage. A `429` *may* indicate a gateway consumption limit, a provider quota, or another rate control: inspect the portal policy and telemetry to identify its source. Other errors are surfaced rather than mislabeled as policy enforcement.

In [ ]:
observations = []
for number in range(1, 4):
    response = requests.post(
        chat_url,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json={'model': alias, 'messages': [{'role': 'user', 'content': 'Reply with one word: hello.'}]},
        timeout=90,
    )
    if response.status_code not in (200, 429):
        response.raise_for_status()
    usage = response.json().get('usage', {}) if response.ok else {}
    observations.append({'request': number, 'status': response.status_code,
                         'prompt_tokens': usage.get('prompt_tokens'),
                         'completion_tokens': usage.get('completion_tokens')})
    print(observations[-1])
    if response.status_code == 429:
        print('Limit/quota reached; check gateway policy and backend quota before attributing the cause.')
        break


## What to show in the portal

Open your gateway's policy/consumption view and telemetry dashboard. Compare request count and token usage for this alias to the observations above; configure a telemetry destination **before** relying on historical usage. Explore estimated model-spend controls, remembering that these estimates differ from actual provider billing. In traditional APIM you can implement AI limits and metrics with explicit API policies; the distinguishing feature here is the AI-specific model/alias and governance management experience, **not** exclusivity of token limiting.